# EDA 핵심 인사이트 요약 (05_integrated_eda.ipynb 기준)

### 원본 파일 (Kaggle, 총 6개 사용)
표본 추출 시 v1과 v2 파일을 모두 합쳐서(중복 제거) 사용했습니다 — v1(과거 데이터 풍부)만으로는 최신 정보가 부족하고, v2(최신 보충)만으로는 과거 데이터가 부족했던 문제를 해결한 방식입니다.

| 파일 | 역할 |
|---|---|
| `train_v2.csv` | 정답(is_churn) |
| `members_v3.csv` | 회원정보 |
| `transactions.csv`(v1) + `transactions_v2.csv`(v2) | 거래 — 두 파일 합쳐서 사용 |
| `user_logs.csv`(v1) + `user_logs_v2.csv`(v2) | 로그 — 두 파일 합쳐서 사용 |

모든 거래·로그는 **2017-02-28 이전** 데이터만 사용 (데이터 누수 방지). 2월 28일 이후 가입한 것으로 확인된 고객은 애초에 모집단에서 제외됨.

### ⚠️ 예전 인사이트와 달라진 점
새 표본은 컷오프를 엄격히 적용하고, 거래/로그 기록이 없는 고객도 배제하지 않고 그대로 포함시켰습니다.          
그 결과 결측치가 늘었지만, 표본이 원본 모집단을 정확히 대표하게 됐습니다 (이탈 비율 9.0%로 원본과 일치).

### 1. Members — 회원 정보
- **`bd`(나이) 이상치**: 0, 음수, 80세 초과가 다수 → `bd_status` 컬럼으로 정상/이상치 구분되어 있음
- **`gender` 결측**: 원래도 절반가량 결측 → 'Unknown' 카테고리로 처리 권장
- **프로필 미입력 개수**: gender·bd·city를 안 채운 정도(0~3개)를 셀수록 이탈률이 달라지는 패턴 확인됨
- **이탈 위험 집단 Top N**: `registered_via`(가입경로) × `tenure_bin`(가입기간) × `gender_status`(성별) 조합 중 이탈률이 특히 높은 세그먼트가 존재      
→ 서비스의 "이탈 원인 제시" 기능과 직결되는 인사이트

### 2. Transactions — 결제 정보
- **`has_tx`(거래 기록 유무)**: 거래 기록이 없는 고객을 절대 조용히 빼면 안 됨 — 이탈률이 다르게 나타남
- **복수 거래 고객군에서 이탈률이 상대적으로 높게 관찰됨** (`multiple_transaction` 관련)
- **할인 결제 고객 이탈률 약 41.0% vs 정상가 이상 결제 고객 약 6.1%** — 다만 할인 거래 표본이 적어 해석 주의 필요 (플랜·자동갱신·취소 여부와 함께 볼 것)
- `last_auto_renew`, `last_is_cancel`, `cancel_on_last_date`, `days_to_expire` 등 "마지막 거래" 기준 파생 변수가 핵심 피처 후보

### 3. User Logs — 청취 로그
- **활동일수(`activity_days`) 구간별 이탈률**: 로그 없음(0일)부터 61일 이상까지 구간별로 이탈률 차이가 뚜렷함
- **마지막 청취 경과일(`days_since_last_log`) 구간별 이탈률**: 최근에 청취할수록 이탈률 낮은 경향
- **"로그 없는 고객의 이탈률이 낮아 보이는" 현상이 자동결제(`last_auto_renew`) 때문인지 교차 확인**됨 — 단순히 "로그 없음 = 이탈 안 함"으로 해석하면 안 됨

### 4. 최종 통합 데이터 컬럼 구성 (integrated_data.csv, 100,000행 × 22컬럼)

**원본 파일에 그대로 있던 컬럼 (6개)**

| 컬럼 | 출처 |
|---|---|
| `msno` | 공통 (고객 ID) |
| `is_churn` | train_v2.csv |
| `city`, `bd`, `gender`, `registered_via`, `registration_init_time` | members_v3.csv |

**transactions에서 파생된 컬럼 (9개)** — 고객당 여러 거래 기록을 1행으로 집계

| 컬럼 | 만든 방식 |
|---|---|
| `transaction_count` | 고객별 거래 건수 |
| `cancel_count` | 고객별 취소 건수 합계 |
| `total_payment` | 취소 안 한 거래의 결제액 합산 |
| `last_auto_renew` | 마지막 거래의 자동갱신 여부 |
| `last_is_cancel` | 마지막 거래의 취소 여부 |
| `last_plan_days` | 마지막 거래의 결제 플랜 일수 |
| `days_to_expire` | 마지막 거래의 만료일 − 기준일(2/28) 차이 |
| `cancel_on_last_date` | 마지막 거래 날짜에 취소 기록이 하나라도 있었는지 |
| `has_transaction` | 거래 기록 자체가 있는지 여부 (0/1) |

**user_logs에서 파생된 컬럼 (6개)** — 고객당 여러 날짜 로그를 1행으로 집계

| 컬럼 | 만든 방식 |
|---|---|
| `activity_days` | 로그를 남긴 날짜 수 |
| `total_secs` | 청취 시간 총합 (하루 86,400초 상한) |
| `total_num_100`, `total_num_unq` | 완청/고유곡 재생 횟수 총합 |
| `days_since_last_log` | 마지막 로그일 − 기준일(2/28) 차이 |
| `has_log` | 로그 기록 자체가 있는지 여부 (0/1) |

**결측치 처리 원칙**: 횟수·합계형 컬럼(transaction_count, cancel_count, total_payment, activity_days, total_secs, total_num_100, total_num_unq)은 이미 0으로 채워져 있음.         
반면 `last_*`, `days_*` 컬럼은 "기록 없음"을 의미하므로 NaN으로 남아있음 → `has_transaction`, `has_log` 플래그로 구분해서 처리 필요

---


# 1. 데이터 로드

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../../data/processed/integrated_data.csv")
print(df.shape)
df.head(10)


(100000, 22)


,msno,is_churn,city,bd,gender,registered_via,registration_init_time,transaction_count,cancel_count,total_payment,...,last_plan_days,days_to_expire,cancel_on_last_date,activity_days,total_secs,total_num_100,total_num_unq,days_since_last_log,has_transaction,has_log
0,UeD6hCJ5rhHxQzrFmzJzCvxX1Y5CCRjMBu9LV//Wwt0=,0,14.0,22.0,female,7.0,20160119.0,14.0,0.0,1386.0,...,30.0,18.0,0.0,4.0,18397.448,68.0,78.0,11.0,1,1
1,Irpqi3F0N1I+H9+JG993AI/SlmHJXpjCLnmK3rTQ2GQ=,0,1.0,0.0,NaN,7.0,20151027.0,17.0,0.0,1683.0,...,30.0,27.0,0.0,0.0,0.000,0.0,0.0,NaN,1,0
2,zF7xywlSpyZ6DI2UI/SQo9GS5PzZlrKja3JcthiqSSU=,0,13.0,29.0,male,7.0,20140518.0,26.0,0.0,3754.0,...,30.0,17.0,0.0,55.0,134757.376,529.0,541.0,3.0,1,1
3,OG33K6iA6t+LB42XJHiL3Cf22/C4f1HfFSz5KGFoKww=,0,1.0,0.0,NaN,4.0,20161214.0,3.0,0.0,450.0,...,30.0,16.0,0.0,65.0,352944.005,1293.0,1312.0,0.0,1,1
4,H1irejyH4SaZ4xszCmwMpDZhuNa4dZl9JijPPqhgGJY=,0,1.0,0.0,NaN,7.0,20130915.0,20.0,0.0,1980.0,...,30.0,6.0,0.0,8.0,29785.691,106.0,121.0,0.0,1,1
5,lg3zG4deV9LDua90hgftaBHgV97OTYNl7iR0IbJDhFw=,0,15.0,29.0,female,9.0,20090718.0,20.0,0.0,2980.0,...,30.0,53.0,0.0,4.0,10133.685,32.0,27.0,75.0,1,1
6,ia7d0Dw3wYAP8evWEwftFCbw0dWoKkrxQ6BQvruSpcs=,0,13.0,34.0,NaN,9.0,20131023.0,22.0,0.0,3278.0,...,30.0,31.0,0.0,23.0,91863.181,324.0,333.0,9.0,1,1
7,Gh2S7y+rTVK3zHwqtGwnAUFxZ4QZo9JuEX01ykl7sDs=,0,1.0,0.0,NaN,7.0,20160122.0,14.0,0.0,1386.0,...,30.0,21.0,0.0,79.0,904878.545,3754.0,2485.0,0.0,1,1
8,j84nJ8grTvXdE3lBR4IZ7Yb3GXnFwD3WvaiZ5S/drfM=,0,11.0,0.0,NaN,3.0,20131217.0,27.0,3.0,3855.0,...,30.0,3.0,0.0,23.0,38083.585,157.0,125.0,0.0,1,1
9,iXWRSn+9nymKIpbu510N//m87vQcnCYUifuXn/GMqGk=,0,1.0,0.0,NaN,7.0,20160104.0,14.0,0.0,1386.0,...,30.0,3.0,0.0,0.0,0.000,0.0,0.0,NaN,1,0


In [2]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 22 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   msno                    100000 non-null  object 
 1   is_churn                100000 non-null  int64  
 2   city                    88648 non-null   float64
 3   bd                      88648 non-null   float64
 4   gender                  39831 non-null   object 
 5   registered_via          88648 non-null   float64
 6   registration_init_time  88648 non-null   float64
 7   transaction_count       100000 non-null  float64
 8   cancel_count            100000 non-null  float64
 9   total_payment           100000 non-null  float64
 10  last_auto_renew         99862 non-null   float64
 11  last_is_cancel          99862 non-null   float64
 12  last_plan_days          99862 non-null   float64
 13  days_to_expire          99862 non-null   float64
 14  cancel_on_last_date  

In [3]:
# 컬럼별 결측치 확인
df.isnull().sum()


msno                          0
is_churn                      0
city                      11352
bd                        11352
gender                    60169
registered_via            11352
registration_init_time    11352
transaction_count             0
cancel_count                  0
total_payment                 0
last_auto_renew             138
last_is_cancel              138
last_plan_days              138
days_to_expire              138
cancel_on_last_date         138
activity_days                 0
total_secs                    0
total_num_100                 0
total_num_unq                 0
days_since_last_log       18281
has_transaction               0
has_log                       0
dtype: int64

In [4]:
# 이탈 비율 확인 (원본과 일치하는지)
df['is_churn'].value_counts(normalize=True)


is_churn
0    0.91049
1    0.08951
Name: proportion, dtype: float64

In [5]:
# has_transaction / has_log 분포 확인 — 결측 처리 방향 결정에 참고
# has_transaction: 거래기록 자체가 있는지
# has_log: 로그기록 자체가 있는지
print(df['has_transaction'].value_counts())
print(df['has_log'].value_counts())


has_transaction
1    99862
0      138
Name: count, dtype: int64
has_log
1    81719
0    18281
Name: count, dtype: int64


# 2. 전처리

### 2-1. 범주형 컬럼 확인

In [6]:
# city, registered_via는 숫자로 저장되어 있지만 실제로는 범주형 변수임
# 원-핫 인코딩 시 컬럼이 얼마나 늘어날지 미리 확인
print(df['city'].nunique(), df['registered_via'].nunique())


21 5


### 2-2. gender 결측치 처리

In [7]:
# gender 결측치를 'Unknown' 카테고리로 채움
# -> 결측 자체가 이탈과 관련 있을 수 있다는 EDA 인사이트를 보존하기 위함
df['gender'] = df['gender'].fillna('Unknown')


### 2-3. bd(나이) 이상치 처리

In [8]:
# EDA에서 확인된 기준: 10~80세만 정상, 그 외(0/음수/81세 이상)는 이상치
print(df['bd'].describe())
print('0살:', (df['bd'] == 0).sum())
print('음수:', (df['bd'] < 0).sum())
print('80세 초과:', (df['bd'] > 80).sum())


count    88648.000000
mean        13.497981
std         19.685454
min        -51.000000
25%          0.000000
50%          0.000000
75%         27.000000
max       1820.000000
Name: bd, dtype: float64
0살: 48995
음수: 7
80세 초과: 59


In [9]:
# EDA에서 확인된 기준: 10~80세만 정상, 그 외(0/음수/81세 이상)는 이상치로 처리
bd_valid = df['bd'].between(10, 80)  # 정상 범위

# 정상 범위 나이의 중앙값으로 이상치를 대체할 기준값 계산
bd_median = df.loc[bd_valid, 'bd'].median()
print("정상 범위 나이의 중앙값:", bd_median)

# bd_is_missing: 나이가 결측(비정상)인지 여부를 별도 플래그로 보존
df['bd_is_missing'] = (~bd_valid).astype(int)

# 비정상 나이(0, 음수, 81세 이상)는 정상 범위 중앙값으로 대체
df['bd'] = df['bd'].where(bd_valid, bd_median)

print(df['bd'].describe())
print("bd_is_missing 비율:\n", df['bd_is_missing'].value_counts(normalize=True))


정상 범위 나이의 중앙값: 28.0
count    100000.000000
mean         28.732860
std           5.602552
min          11.000000
25%          28.000000
50%          28.000000
75%          28.000000
max          80.000000
Name: bd, dtype: float64
bd_is_missing 비율:
 bd_is_missing
1    0.60424
0    0.39576
Name: proportion, dtype: float64


### 2-4. 거래/로그 없는 고객 처리 (has_transaction, has_log 활용)

In [10]:
# last_auto_renew, last_is_cancel 등은 거래 기록이 없으면(has_transaction=0) NaN
# -1로 채워서 "기록 없음"을 별도 값으로 표시 (0/1과 겹치지 않는 값)
df['last_auto_renew'] = df['last_auto_renew'].fillna(-1)
df['last_is_cancel'] = df['last_is_cancel'].fillna(-1)
df['last_plan_days'] = df['last_plan_days'].fillna(-1)
df['cancel_on_last_date'] = df['cancel_on_last_date'].fillna(-1)

# days_to_expire: '마지막 거래 만료일 - 기준일(2/28)' 파생 컬럼
# -> is_churn 그룹별 분포/음수비율 확인 + LightGBM feature importance 1위 + 제거 시 ROC-AUC 3.77%p 하락 확인됨
# -> 이탈 여부와 사실상 직결된 정보(정답 유출)로 판단해 모델링에서 완전히 제외함
df = df.drop(columns=['days_to_expire'])

# days_since_last_log: 로그가 없는 사람은 "아주 오랫동안 안 들었다"는 의미로 최댓값으로 채움
df['days_since_last_log'] = df['days_since_last_log'].fillna(df['days_since_last_log'].max())

# 처리 후 결측치 재확인
df.isnull().sum()


msno                          0
is_churn                      0
city                      11352
bd                            0
gender                        0
registered_via            11352
registration_init_time    11352
transaction_count             0
cancel_count                  0
total_payment                 0
last_auto_renew               0
last_is_cancel                0
last_plan_days                0
cancel_on_last_date           0
activity_days                 0
total_secs                    0
total_num_100                 0
total_num_unq                 0
days_since_last_log           0
has_transaction               0
has_log                       0
bd_is_missing                 0
dtype: int64

### 2-5. 범주형 컬럼 원-핫 인코딩

In [11]:
# city, registered_via 결측치를 별도 카테고리(-1)로 채움
df['city'] = df['city'].fillna(-1)
df['registered_via'] = df['registered_via'].fillna(-1)

df.isnull().sum()

msno                          0
is_churn                      0
city                          0
bd                            0
gender                        0
registered_via                0
registration_init_time    11352
transaction_count             0
cancel_count                  0
total_payment                 0
last_auto_renew               0
last_is_cancel                0
last_plan_days                0
cancel_on_last_date           0
activity_days                 0
total_secs                    0
total_num_100                 0
total_num_unq                 0
days_since_last_log           0
has_transaction               0
has_log                       0
bd_is_missing                 0
dtype: int64

In [12]:
# city, registered_via, gender 원-핫 인코딩
# drop_first=True: 다중공선성 방지를 위해 각 컬럼당 카테고리 1개는 기준값으로 제외
categorical_cols = ['gender', 'city', 'registered_via']
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# registration_init_time은 문자열(날짜) 그대로 모델에 넣을 수 없으므로 제거
# (필요하면 가입 후 경과일수 등의 파생변수로 대체 가능)
df_encoded = df_encoded.drop(columns=['registration_init_time'])

print(df_encoded.shape)
df_encoded.head()


(100000, 46)


,msno,is_churn,bd,transaction_count,cancel_count,total_payment,last_auto_renew,last_is_cancel,last_plan_days,cancel_on_last_date,...,city_18.0,city_19.0,city_20.0,city_21.0,city_22.0,registered_via_3.0,registered_via_4.0,registered_via_7.0,registered_via_9.0,registered_via_13.0
0,UeD6hCJ5rhHxQzrFmzJzCvxX1Y5CCRjMBu9LV//Wwt0=,0,22.0,14.0,0.0,1386.0,1.0,0.0,30.0,0.0,...,False,False,False,False,False,False,False,True,False,False
1,Irpqi3F0N1I+H9+JG993AI/SlmHJXpjCLnmK3rTQ2GQ=,0,28.0,17.0,0.0,1683.0,1.0,0.0,30.0,0.0,...,False,False,False,False,False,False,False,True,False,False
2,zF7xywlSpyZ6DI2UI/SQo9GS5PzZlrKja3JcthiqSSU=,0,29.0,26.0,0.0,3754.0,1.0,0.0,30.0,0.0,...,False,False,False,False,False,False,False,True,False,False
3,OG33K6iA6t+LB42XJHiL3Cf22/C4f1HfFSz5KGFoKww=,0,28.0,3.0,0.0,450.0,0.0,0.0,30.0,0.0,...,False,False,False,False,False,False,True,False,False,False
4,H1irejyH4SaZ4xszCmwMpDZhuNa4dZl9JijPPqhgGJY=,0,28.0,20.0,0.0,1980.0,1.0,0.0,30.0,0.0,...,False,False,False,False,False,False,False,True,False,False


# 3. train_test_split

In [13]:
from sklearn.model_selection import train_test_split

# X(피처)와 y(타겟) 분리
# msno는 식별자라 모델 학습에 불필요하므로 제외, is_churn은 예측 대상이므로 제외
X = df_encoded.drop(columns=['msno', 'is_churn'])
y = df_encoded['is_churn']

# 공통 규칙: test_size=0.2, stratify=y(클래스 비율 유지), random_state=42(재현성)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
X_train.shape, X_test.shape


((80000, 44), (20000, 44))

# 4. 베이스라인 모델

In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, classification_report

# LogisticRegression은 피처 값의 범위 차이에 민감 -> 스케일링 적용
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# class_weight='balanced': 이탈(1)이 소수 클래스이므로 가중치 자동 부여
log_model = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
log_model.fit(X_train_scaled, y_train)

rf_model = RandomForestClassifier(class_weight='balanced', random_state=42)
rf_model.fit(X_train, y_train)

print("학습 완료")


학습 완료


# 5. 평가

In [15]:
log_pred_proba = log_model.predict_proba(X_test_scaled)[:, 1]
log_pred = log_model.predict(X_test_scaled)

print("=== Logistic Regression ===")
print("ROC-AUC:", roc_auc_score(y_test, log_pred_proba))
print(classification_report(y_test, log_pred))

rf_pred_proba = rf_model.predict_proba(X_test)[:, 1]
rf_pred = rf_model.predict(X_test)

print("\n=== Random Forest ===")
print("ROC-AUC:", roc_auc_score(y_test, rf_pred_proba))
print(classification_report(y_test, rf_pred))


=== Logistic Regression ===
ROC-AUC: 0.8227412650057215
              precision    recall  f1-score   support

           0       0.96      0.92      0.94     18210
           1       0.44      0.65      0.52      1790

    accuracy                           0.89     20000
   macro avg       0.70      0.78      0.73     20000
weighted avg       0.92      0.89      0.90     20000


=== Random Forest ===
ROC-AUC: 0.8355896600492699
              precision    recall  f1-score   support

           0       0.95      0.96      0.96     18210
           1       0.56      0.54      0.55      1790

    accuracy                           0.92     20000
   macro avg       0.76      0.75      0.75     20000
weighted avg       0.92      0.92      0.92     20000



# 6. 부스팅 모델 비교 (XGBoost, LightGBM, CatBoost)

In [16]:
import xgboost as xgb
import lightgbm as lgb
import catboost as cb

# 클래스 불균형 대응: 양성(이탈) 클래스에 가중치 부여
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb_model = xgb.XGBClassifier(
    scale_pos_weight=scale_pos_weight, random_state=42, eval_metric='auc'
)
xgb_model.fit(X_train, y_train)

lgb_model = lgb.LGBMClassifier(
    scale_pos_weight=scale_pos_weight, random_state=42, verbose=-1
)
lgb_model.fit(X_train, y_train)

cb_model = cb.CatBoostClassifier(
    scale_pos_weight=scale_pos_weight, random_state=42, verbose=0
)
cb_model.fit(X_train, y_train)

print("학습 완료 (XGBoost, LightGBM, CatBoost)")

학습 완료 (XGBoost, LightGBM, CatBoost)


In [17]:
import pandas as pd

results = {
    'Logistic Regression': roc_auc_score(y_test, log_pred_proba),
    'Random Forest': roc_auc_score(y_test, rf_pred_proba),
    'XGBoost': roc_auc_score(y_test, xgb_model.predict_proba(X_test)[:, 1]),
    'LightGBM': roc_auc_score(y_test, lgb_model.predict_proba(X_test)[:, 1]),
    'CatBoost': roc_auc_score(y_test, cb_model.predict_proba(X_test)[:, 1]),
}

pd.Series(results).sort_values(ascending=False).to_frame('ROC-AUC')

,ROC-AUC
LightGBM,0.860269
CatBoost,0.850358
XGBoost,0.847848
Random Forest,0.835590
Logistic Regression,0.822741


# 7. 최종 정리

### 이번 버전에서 바뀐 점
- `days_to_expire`(마지막 거래 만료일 − 기준일(2/28)) 컬럼을 **모델링에서 완전히 제외**함
  - 근거: is_churn별 음수 비율이 유지 0.17% vs 이탈 8.76%로 크게 차이남, LightGBM feature importance 45개 중 1위(단독 1위),      
  제거 시 LightGBM ROC-AUC가 0.8980 → 0.8603로 3.77%p 하락 — 데이터 누수(정답 유출) 가능성이 높다고 판단함 
  - 이전에 발견됐던 "마지막 거래/활동 기준" 데이터 누수(2017년 3월 데이터 84.6~100% 포함)와 같은 계열의 문제로 판단

### 전처리 요약
1. `gender` 결측 → 'Unknown' 카테고리
2. `bd`(나이) 이상치(0/음수/80세 초과) → 정상범위(10~80세) 중앙값으로 대체, `bd_is_missing` 플래그 추가
3. `last_auto_renew`, `last_is_cancel`, `last_plan_days`, `cancel_on_last_date` 결측(거래 기록 없음) → -1
4. `days_to_expire` → **컬럼 자체를 제거** (데이터 누수 판단)
5. `days_since_last_log` 결측(로그 없음) → 최댓값
6. `city`, `registered_via` 결측 → -1 별도 카테고리
7. `gender`, `city`, `registered_via` 원-핫 인코딩 (drop_first=True)
8. `registration_init_time` 제거 (날짜 문자열, 모델 입력 불가)

### 모델링 설정
- `train_test_split(test_size=0.2, stratify=y, random_state=42)`
- 비교 모델: Logistic Regression, Random Forest, XGBoost, LightGBM, CatBoost
- 클래스 불균형 대응: Logistic/RF는 `class_weight='balanced'`, 부스팅 3종은 `scale_pos_weight`
